# Project 1 — Data Cleaning & Preparation
### DecodeLabs | Data Analytics Internship — Batch 2026

**Analyst:** Intern, DecodeLabs Data Analytics Track
**Dataset:** E-Commerce Order Transactions (`ecommerce_orders_raw.xlsx`)
**Objective:** Audit a raw transactional dataset for missing values, duplicate
records, and inconsistent formatting, then produce a validated, analysis-ready
"gold standard" dataset — following the four-phase workflow defined in the
project brief:

1. **Phase 1 — Strategic Imputation:** Handle missing values without blind deletion.
2. **Phase 2 — Integrity Audit:** Guarantee one truth, one record (deduplication).
3. **Phase 3 — Standardization:** Enforce a single consistent format across dates, text, and numbers.
4. **Phase 4 — Verification Gate:** Prove 0% error rate on unique identifiers and date formats before sign-off.

Every change made below is logged in [`reports/change_log.md`](../reports/change_log.md)
so the transformation is fully reproducible and auditable.


## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

RAW_PATH = Path('../data/raw/ecommerce_orders_raw.xlsx')
CLEAN_PATH = Path('../data/cleaned/ecommerce_orders_cleaned.csv')

print("Environment ready.")


Environment ready.


## 2. Load the Raw Dataset

In [2]:
df = pd.read_excel(RAW_PATH)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()


Rows: 1,200 | Columns: 14


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    str           
 12  ReferralSource   1200 non-null   str           
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(9)


## 3. Data Quality Audit
Before touching a single value, we quantify exactly what is wrong with the
data. This audit result is what the cleaning steps in Section 4 will be
graded against (Section 6 re-runs the same checks to prove they now pass).


### 3.1 Missing Value Audit

In [4]:
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2)
})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)
missing_summary


,missing_count,missing_pct
CouponCode,309,25.75


### 3.2 Duplicate Record Audit
`OrderID` is the primary key — it must be 100% unique. We check both full-row duplicates and duplicate keys.

In [5]:
full_row_dupes = df.duplicated().sum()
orderid_dupes = df['OrderID'].duplicated().sum()

print(f"Full-row duplicates : {full_row_dupes}")
print(f"Duplicate OrderIDs  : {orderid_dupes}")


Full-row duplicates : 0
Duplicate OrderIDs  : 0


### 3.3 Formatting Consistency Audit
Checking identifier patterns, date formats, and text casing/whitespace across every categorical column.

In [6]:
id_patterns = {
    'OrderID': r'^ORD\d{6}$',
    'CustomerID': r'^C\d{5}$',
    'TrackingNumber': r'^TRK\d{8}$',
}

print("--- Identifier format violations ---")
for col, pattern in id_patterns.items():
    bad = (~df[col].astype(str).str.match(pattern)).sum()
    print(f"{col:<15}: {bad} malformed value(s)")

print("\n--- Date format violations ---")
bad_dates = pd.to_datetime(df['Date'], errors='coerce').isna().sum()
print(f"Date            : {bad_dates} unparseable value(s)")

print("\n--- Whitespace violations (leading/trailing spaces) ---")
text_cols = df.select_dtypes(include='object').columns.tolist() + \
            [c for c in df.columns if str(df[c].dtype) == 'string']
text_cols = sorted(set(text_cols))
for col in text_cols:
    ws = df[col].dropna().astype(str).apply(lambda x: x != x.strip()).sum()
    print(f"{col:<17}: {ws} value(s) with stray whitespace")


--- Identifier format violations ---
OrderID        : 0 malformed value(s)
CustomerID     : 0 malformed value(s)
TrackingNumber : 0 malformed value(s)

--- Date format violations ---
Date            : 0 unparseable value(s)

--- Whitespace violations (leading/trailing spaces) ---
CouponCode       : 0 value(s) with stray whitespace
CustomerID       : 0 value(s) with stray whitespace
OrderID          : 0 value(s) with stray whitespace
OrderStatus      : 0 value(s) with stray whitespace
PaymentMethod    : 0 value(s) with stray whitespace
Product          : 0 value(s) with stray whitespace
ReferralSource   : 0 value(s) with stray whitespace
ShippingAddress  : 0 value(s) with stray whitespace
TrackingNumber   : 0 value(s) with stray whitespace


/tmp/ipykernel_632/1649896896.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include='object').columns.tolist() + \


In [7]:
print("--- Categorical value inventory (checking for case/spelling inconsistency) ---")
for col in ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'CouponCode']:
    print(f"\n{col}: {sorted(df[col].dropna().unique().tolist())}")


--- Categorical value inventory (checking for case/spelling inconsistency) ---

Product: ['Chair', 'Desk', 'Laptop', 'Monitor', 'Phone', 'Printer', 'Tablet']

PaymentMethod: ['Cash', 'Credit Card', 'Debit Card', 'Gift Card', 'Online']

OrderStatus: ['Cancelled', 'Delivered', 'Pending', 'Returned', 'Shipped']

ReferralSource: ['Email', 'Facebook', 'Google', 'Instagram', 'Referral']

CouponCode: ['FREESHIP', 'SAVE10', 'WINTER15']


### 3.4 Business-Logic Consistency Check
`TotalPrice` should always equal `Quantity × UnitPrice`. Any mismatch signals
a data-entry or calculation error upstream.


In [8]:
expected_total = (df['Quantity'] * df['UnitPrice']).round(2)
mismatches = (expected_total != df['TotalPrice'].round(2)).sum()
print(f"TotalPrice calculation mismatches: {mismatches}")


TotalPrice calculation mismatches: 0


### 3.5 Audit Summary

| Check | Result |
|---|---|
| Missing values | `CouponCode` has nulls — **not an error**, it means no coupon was applied at checkout |
| Full-row duplicates | 0 found |
| Duplicate `OrderID` (primary key) | 0 found |
| Identifier format violations | 0 found |
| Unparseable dates | 0 found |
| Stray whitespace | 0 found |
| `TotalPrice` calculation mismatches | 0 found |

The raw feed is structurally sound, but `CouponCode` nulls are ambiguous —
a `NaN` could be misread by downstream tools as "unknown" rather than
"no coupon used." Section 4 resolves this and hardens every column against
future format drift with defensive, re-runnable cleaning logic (so the same
notebook stays correct even if a messier data pull arrives next time).


## 4. Cleaning & Transformation Pipeline
Each step below is intentionally defensive: it *proves* a class of error is
absent (or fixes it) rather than assuming the data is fine. This is what
makes the pipeline reusable on future, messier data pulls.


### Phase 1 — Strategic Imputation (Missing Values)
`CouponCode` nulls are recoded to the explicit category `'NoCoupon'`. This preserves every row (no listwise deletion, which would destroy statistical power) and removes ambiguity for downstream analysis.

In [9]:
df_clean = df.copy()

df_clean['CouponCode'] = df_clean['CouponCode'].fillna('NoCoupon')

print(f"Remaining nulls in CouponCode: {df_clean['CouponCode'].isnull().sum()}")
df_clean['CouponCode'].value_counts()


Remaining nulls in CouponCode: 0


CouponCode
FREESHIP    313
NoCoupon    309
WINTER15    292
SAVE10      286
Name: count, dtype: int64

### Phase 2 — Integrity Audit (De-duplication)
Drop any full-row duplicates and any repeated `OrderID` (keeping the first occurrence), so every primary key maps to exactly one record.

In [10]:
rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset='OrderID', keep='first')

rows_after = len(df_clean)
print(f"Rows before: {rows_before} | Rows after: {rows_after} | Removed: {rows_before - rows_after}")


Rows before: 1200 | Rows after: 1200 | Removed: 0


### Phase 3 — Standardization (Dates, Text, Numbers)
- Dates → ISO 8601 (`YYYY-MM-DD`)
- Text columns → whitespace trimmed, consistent title case
- Monetary columns → rounded to 2 decimal places

In [11]:
# Dates -> ISO 8601
df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Text standardization: trim whitespace + consistent casing
categorical_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'CouponCode']
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

df_clean['ShippingAddress'] = df_clean['ShippingAddress'].astype(str).str.strip()
df_clean['OrderID'] = df_clean['OrderID'].astype(str).str.strip().str.upper()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(str).str.strip().str.upper()
df_clean['TrackingNumber'] = df_clean['TrackingNumber'].astype(str).str.strip().str.upper()

# Numeric precision -> 2 decimals
df_clean['UnitPrice'] = df_clean['UnitPrice'].round(2)
df_clean['TotalPrice'] = df_clean['TotalPrice'].round(2)

df_clean.head()


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,Save10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,Save10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,Freeship,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,Save10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,Save10,Email,2504.04


### Phase 3b — Recalculate & Repair `TotalPrice`
Even though no mismatches were found in the audit, we defensively recompute `TotalPrice` from `Quantity × UnitPrice` so the pipeline self-heals if this ever drifts in a future data pull.

In [12]:
df_clean['TotalPrice'] = (df_clean['Quantity'] * df_clean['UnitPrice']).round(2)
print("TotalPrice recalculated and locked to Quantity x UnitPrice for all rows.")


TotalPrice recalculated and locked to Quantity x UnitPrice for all rows.


## 5. Verification Gate
Per the project brief: *"Before you finish, you must prove there are zero
duplicate IDs and zero incorrectly formatted dates."* We re-run the exact
audit from Section 3 against the cleaned dataset and assert every check
passes.


In [13]:
checks = {}

checks['Duplicate OrderIDs']       = df_clean['OrderID'].duplicated().sum()
checks['Full-row duplicates']      = df_clean.duplicated().sum()
checks['Malformed OrderID']        = (~df_clean['OrderID'].str.match(r'^ORD\d{6}$')).sum()
checks['Malformed CustomerID']     = (~df_clean['CustomerID'].str.match(r'^C\d{5}$')).sum()
checks['Malformed TrackingNumber'] = (~df_clean['TrackingNumber'].str.match(r'^TRK\d{8}$')).sum()
checks['Unparseable / bad dates']  = (~df_clean['Date'].str.match(r'^\d{4}-\d{2}-\d{2}$')).sum()
checks['Remaining nulls (any col)']= df_clean.isnull().sum().sum()
checks['TotalPrice mismatches']    = ((df_clean['Quantity'] * df_clean['UnitPrice']).round(2) != df_clean['TotalPrice']).sum()

result = pd.DataFrame.from_dict(checks, orient='index', columns=['error_count'])
result['status'] = result['error_count'].apply(lambda x: 'PASS' if x == 0 else 'FAIL')
result


,error_count,status
Duplicate OrderIDs,0,PASS
Full-row duplicates,0,PASS
Malformed OrderID,0,PASS
Malformed CustomerID,0,PASS
Malformed TrackingNumber,0,PASS
Unparseable / bad dates,0,PASS
Remaining nulls (any col),0,PASS
TotalPrice mismatches,0,PASS


In [14]:
assert result['error_count'].sum() == 0, "Verification gate FAILED — unresolved data quality issues remain."
print("VERIFICATION GATE: PASSED — 0% error rate across all checks. Dataset is production-ready.")


VERIFICATION GATE: PASSED — 0% error rate across all checks. Dataset is production-ready.


## 6. Before vs. After Summary

In [15]:
summary = pd.DataFrame({
    'Raw Dataset': [
        df.shape[0],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        int(df['OrderID'].duplicated().sum()),
    ],
    'Cleaned Dataset': [
        df_clean.shape[0],
        int(df_clean.isnull().sum().sum()),
        int(df_clean.duplicated().sum()),
        int(df_clean['OrderID'].duplicated().sum()),
    ],
}, index=['Row Count', 'Missing Values', 'Duplicate Rows', 'Duplicate OrderIDs'])

summary


,Raw Dataset,Cleaned Dataset
Row Count,1200,1200
Missing Values,309,0
Duplicate Rows,0,0
Duplicate OrderIDs,0,0


## 7. Export the Cleaned, Analysis-Ready Dataset

In [16]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEAN_PATH.resolve().name}")
print(f"Final shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")


Cleaned dataset saved to: ecommerce_orders_cleaned.csv
Final shape: 1,200 rows x 14 columns


## 8. Conclusion

The raw `ecommerce_orders_raw.xlsx` feed (1,200 rows) has been audited and
transformed into a verified, analysis-ready dataset:

- **0** duplicate `OrderID` values (primary key integrity guaranteed)
- **0** malformed identifiers, dates, or unresolved nulls
- `CouponCode` gaps resolved to an explicit `'NoCoupon'` category instead of being silently dropped
- Dates standardized to **ISO 8601**, text fields trimmed and case-normalized, monetary fields locked to 2-decimal precision
- `TotalPrice` is now guaranteed consistent with `Quantity x UnitPrice` for every row

The output file, [`ecommerce_orders_cleaned.csv`](../data/cleaned/ecommerce_orders_cleaned.csv),
is ready to feed into Project 2 (Exploratory Data Analysis & Dashboards).
Full change history is documented in [`reports/change_log.md`](../reports/change_log.md).

---
**DecodeLabs | Data Analytics Internship — Project 1 of the Industrial Training Kit**
